# COLMAP dense reconstruction (Colab, GPU) + metrics

Takes `colmap_sparse.zip` produced by the local no-GPU notebook (unknown-pose sparse
model: `images/`, `masks/`, `sparse/0`) and runs the GPU dense stage: `image_undistorter`
-> `patch_match_stereo` -> `stereo_fusion`, then reports quantitative and qualitative
results and saves everything to Drive.

**Runtime:** make sure you're on a GPU runtime (Runtime -> Change runtime type -> T4 GPU)
before running Cell 1.

## Cell 1 — Install COLMAP

In [1]:
!apt-get update -qq
!apt-get install -y colmap

!pip install -q condacolab
import condacolab
condacolab.install()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libamd2 libatk-bridge2.0-0
  libatk1.0-0 libatk1.0-data libatspi2.0-0 libcamd2 libccolamd2 libceres2
  libcholmod3 libcolamd2 libcxsparse3 libdouble-conversion3 libevdev2
  libfreeimage3 libgflags2.2 libglew2.2 libgoogle-glog0v5 libgtk-3-0
  libgtk-3-bin libgtk-3-common libgudev-1.0-0 libilmbase25 libinput-bin
  libinput10 libjxr0 libmd4c0 libmetis5 libmtdev1 libopenexr25 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libraw20
  librsvg2-common libspqr2 libsuitesparseconfig5 libwacom-bin libwacom-common
  libwacom9 libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0
  l

In [2]:
!rm -f /usr/local/conda-meta/pinned
!cat /usr/local/conda-meta/pinned 2>/dev/null || echo "pin file removed"

!mamba install -y -c conda-forge colmap "python=3.11"

!mamba remove -y colmap faiss faiss-gpu 2>/dev/null
!mamba install -y -c conda-forge colmap

!mamba create -y -n colmap_env -c conda-forge colmap

!mamba install -y -n colmap_env -c conda-forge faiss

!mamba env remove -y -n colmap_env
!mamba create -y -n colmap_env -c conda-forge python=3.11 colmap faiss

!mamba env remove -y -n colmap_env
!mamba create -y -n colmap_env -c conda-forge python=3.11 colmap faiss openimageio

pin file removed

Looking for: ['colmap', 'python=3.11']

[+] 0.0s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  [+] 0.1s
conda-forge/linux-64   1%
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64   7%
conda-forge/noarch     8%[+] 0.3s
conda-forge/linux-64  15%
conda-forge/noarch    26%[+] 0.4s
conda-forge/linux-64  21%
conda-forge/noarch    40%[+] 0.5s
conda-forge/linux-64  23%
conda-forge/noarch    52%[+] 0.6s
conda-forge/linux-64  30%
conda-forge/noarch    65%[+] 0.7s
conda-forge/linux-64  37%
conda-forge/noarch    79%[+] 0.8s
conda-forge/linux-64  44%
conda-forge/noarch    93%[+] 0.9s
conda-forge/linux-64  49%
conda-forge/noarch    97%conda-forge/noarch                                
[+] 1.0s
conda-forge/linux-64  51%[+] 1.1s
conda-forge/linux-64  56%[+] 1.2s
conda-forge/linux-64  60%[+] 1.3s
conda-forge/linux-64  68%[+] 1.4s
conda-forge/linux-64  75%[+] 1.5s
conda-forge/linux-64  77%[+] 1.6s
conda-forge/linux-64  80%[+] 1.7s
conda-forge/linux-64  80%[+] 1.8s
conda-forg

In [3]:
!mamba run -n colmap_env colmap patch_match_stereo --help | grep -i cuda

!mamba run -n colmap_env ldd $(mamba run -n colmap_env which colmap) | grep "not found"

I20260723 07:42:29.176847 136242898526208 option_manager.cc:1214] COLMAP 4.1.1 (Commit Unknown on Unknown with CUDA)
I20260723 07:42:29.177190 136242898526208 option_manager.cc:1216] Options can either be specified via command-line or by defining them in a .ini project file passed to `--project_path`.
  -h [ --help ] 
  --project_path arg
  --default_random_seed arg (=0)
  --log_target arg (=stderr_and_file)   {stderr, stdout, file, stderr_and_file}
  --log_path arg
  --log_level arg (=0)
  --log_severity arg (=0)               0:INFO, 1:WARNING, 2:ERROR, 3:FATAL
  --log_color arg (=1)
  --workspace_path arg                  Path to the folder containing the 
                                        undistorted images
  --workspace_format arg (=COLMAP)      {COLMAP, PMVS}
  --pmvs_option_name arg (=option-all)
  --config_path arg
  --PatchMatchStereo.max_image_size arg (=-1)
  --PatchMatchStereo.gpu_index arg (=-1)
  --PatchMatchStereo.depth_min arg (=-1)
  --PatchMatchStereo.depth_max 

## Cell 2 — Upload colmap_sparse.zip

In [1]:
from google.colab import files
uploaded = files.upload()

Saving plant3_colmap_sparse.zip to plant3_colmap_sparse.zip


## Cell 3 — Unzip + sanity check the folder layout

In [6]:
!unzip -q plant3_colmap_sparse.zip -d /content/plant3_colmap_sparse
!echo "--- images ---" && ls /content/plant3_colmap_sparse/images | wc -l
!echo "--- masks ---"  && ls /content/plant3_colmap_sparse/masks | wc -l
!echo "--- sparse ---" && ls /content/plant3_colmap_sparse/sparse/0

--- images ---
139
--- masks ---
139
--- sparse ---
cameras.bin  images.bin  points3D.bin  project.ini


## Cell 4 — Run patch_match_stereo (the GPU step)

In [7]:
!nvidia-smi

Thu Jul 23 07:50:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!mamba run -n colmap_env colmap image_undistorter \
    --image_path /content/plant3_colmap_sparse/images \
    --input_path /content/plant3_colmap_sparse/sparse/0 \
    --output_path /content/dense \
    --output_type COLMAP

!mamba run -n colmap_env colmap patch_match_stereo \
    --workspace_path /content/dense \
    --workspace_format COLMAP \
    --PatchMatchStereo.geom_consistency true \
    --PatchMatchStereo.filter true \
    --PatchMatchStereo.num_iterations 8 \
    --PatchMatchStereo.window_radius 9 \
    --PatchMatchStereo.num_samples 20

I20260723 08:28:28.127438 138208560738304 image.cc:368] === Reading reconstruction ===
I20260723 08:28:28.187493 138208560738304 image.cc:371] => Reconstruction with 139 images and 21787 points
I20260723 08:28:28.187574 138208560738304 undistorters.cc:166] === Image undistortion ===
I20260723 08:28:28.188306 138208560738304 undistorters.cc:205] Undistorting image [1/139]
I20260723 08:28:28.681369 138208560738304 undistorters.cc:205] Undistorting image [2/139]
I20260723 08:28:28.681408 138208560738304 undistorters.cc:205] Undistorting image [3/139]
I20260723 08:28:29.124123 138208560738304 undistorters.cc:205] Undistorting image [4/139]
I20260723 08:28:29.124156 138208560738304 undistorters.cc:205] Undistorting image [5/139]
I20260723 08:28:29.597208 138208560738304 undistorters.cc:205] Undistorting image [6/139]
I20260723 08:28:29.597254 138208560738304 undistorters.cc:205] Undistorting image [7/139]
I20260723 08:28:30.042590 138208560738304 undistorters.cc:205] Undistorting image [8/1

## Fuse into dense point cloud

In [ ]:
!mamba run -n colmap_env colmap stereo_fusion \
    --workspace_path /content/dense \
    --workspace_format COLMAP \
    --input_type geometric \
    --output_path /content/dense/fused_plant3.ply \
    --StereoFusion.min_num_pixels 2 \
    --StereoFusion.max_reproj_error 2 \
    --StereoFusion.max_depth_error 0.01 \
    --StereoFusion.max_normal_error 15

## Zip results and download

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_DIR = "/content/drive/MyDrive/JetCobot_internship_2026/colmap_results"
import os
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

!zip -r dense_result.zip /content/dense/stereo /content/dense/fused.ply
!cp dense_result.zip "{DRIVE_SAVE_DIR}/dense_result.zip"

## Cell 5 — Quantitative metrics

Registered images, point counts, mean track length, and mean reprojection error on both
the uploaded sparse model and the undistorted one, plus fused-point counts and
point-per-image density from the dense stage.

In [ ]:
import re
import subprocess

def analyze_model(path):
    result = subprocess.run(
        ["mamba", "run", "-n", "colmap_env", "colmap", "model_analyzer", "--path", path],
        check=True, capture_output=True, text=True,
    )
    log = result.stderr
    def grab(pattern, cast=int):
        m = re.search(pattern, log)
        return cast(m.group(1)) if m else None
    return {
        "registered_images": grab(r"Registered images:\s*(\d+)"),
        "points": grab(r"Points:\s*(\d+)"),
        "mean_track_length": grab(r"Mean track length:\s*([\d.]+)", float),
        "mean_reproj_error_px": grab(r"Mean reprojection error:\s*([\d.]+)px", float),
    }

# the sparse model you uploaded (poses estimated locally, unknown-pose pipeline)
sparse_metrics = analyze_model("/content/colmap_sparse/sparse/0")
# the same model after image_undistorter re-writes it for the dense/patch-match stage
dense_sparse_metrics = analyze_model("/content/dense/sparse")

print("Uploaded sparse model:      ", sparse_metrics)
print("Undistorted (dense) model:  ", dense_sparse_metrics)

In [ ]:
def ply_vertex_count(ply_path):
    with open(ply_path, "rb") as f:
        for line in f:
            line = line.strip()
            if line.startswith(b"element vertex"):
                return int(line.split()[-1])
            if line == b"end_header":
                break
    return None

num_images = len(os.listdir("/content/colmap_sparse/images"))
fused_points = ply_vertex_count("/content/dense/fused.ply")
fused_points_loose = ply_vertex_count("/content/dense/fused_plant22.ply")
density_per_image = (fused_points / num_images) if fused_points else None

metrics_report = f"""COLMAP reconstruction metrics
==============================
Input images                        : {num_images}
Sparse model (uploaded)             : {sparse_metrics}
Sparse model (post-undistortion)    : {dense_sparse_metrics}
Dense fused points (masked, strict) : {fused_points}
Dense fused points (loose, no mask) : {fused_points_loose}
Density (points / registered image) : {f'{density_per_image:.1f}' if density_per_image else 'n/a'}
"""
print(metrics_report)

with open("/content/dense/metrics_report.txt", "w") as f:
    f.write(metrics_report)

## Cell 6 — Qualitative results

Fused point cloud with the recovered camera trajectory overlaid, plus a strip of sample
input frames for a quick visual sanity check.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

def read_ply_xyz(path, max_points=200_000):
    with open(path, "rb") as f:
        header = []
        while True:
            line = f.readline()
            header.append(line)
            if line.strip() == b"end_header":
                break
        n_verts, props = None, []
        for h in header:
            if h.startswith(b"element vertex"):
                n_verts = int(h.split()[-1])
            elif h.startswith(b"property") and n_verts is not None:
                props.append(h.split()[-1].decode())
        dtype = np.dtype([
            (p, "<f4") if p in ("x", "y", "z", "nx", "ny", "nz") else (p, "<u1")
            for p in props
        ])
        data = np.fromfile(f, dtype=dtype, count=n_verts)
    xyz = np.stack([data["x"], data["y"], data["z"]], axis=1)
    if len(xyz) > max_points:
        idx = np.random.choice(len(xyz), max_points, replace=False)
        xyz = xyz[idx]
    return xyz

points = read_ply_xyz("/content/dense/fused.ply")

In [ ]:
!mamba run -n colmap_env colmap model_converter \
    --input_path /content/dense/sparse \
    --output_path /content/dense/sparse_txt \
    --output_type TXT

In [ ]:
centers = []
with open("/content/dense/sparse_txt/images.txt") as f:
    lines = [l for l in f if l.strip() and not l.startswith("#")]
for line in lines[::2]:  # each image occupies 2 lines; the pose is on the first
    parts = line.split()
    qw, qx, qy, qz, tx, ty, tz = map(float, parts[1:8])
    R = np.array([
        [1 - 2*qy**2 - 2*qz**2, 2*qx*qy - 2*qz*qw,     2*qx*qz + 2*qy*qw],
        [2*qx*qy + 2*qz*qw,     1 - 2*qx**2 - 2*qz**2, 2*qy*qz - 2*qx*qw],
        [2*qx*qz - 2*qy*qw,     2*qy*qz + 2*qx*qw,     1 - 2*qx**2 - 2*qy**2],
    ])
    t = np.array([tx, ty, tz])
    centers.append(-R.T @ t)
centers = np.array(centers)

fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(points[:, 0], points[:, 1], points[:, 2], s=0.3, c="gray", alpha=0.5, label="fused points")
ax.scatter(centers[:, 0], centers[:, 1], centers[:, 2], s=20, c="red", label="camera centers")
ax.plot(centers[:, 0], centers[:, 1], centers[:, 2], c="red", linewidth=0.8, alpha=0.6)
ax.set_title("Fused point cloud + recovered camera trajectory")
ax.legend()
plt.tight_layout()
plt.savefig("/content/dense/qualitative_reconstruction.png", dpi=150)
plt.show()

In [ ]:
# sample input frames, as a visual sanity check next to the reconstruction above
sample_files = sorted(os.listdir("/content/colmap_sparse/images"))[::max(1, num_images // 6)][:6]
fig, axes = plt.subplots(1, len(sample_files), figsize=(3 * len(sample_files), 3))
for ax_i, fname in zip(axes, sample_files):
    img = plt.imread(os.path.join("/content/colmap_sparse/images", fname))
    ax_i.imshow(img)
    ax_i.set_title(fname, fontsize=8)
    ax_i.axis("off")
plt.tight_layout()
plt.savefig("/content/dense/sample_frames.png", dpi=150)
plt.show()

## Cell 7 — Save metrics + figures to Drive

Alongside `dense_result.zip`, saved earlier.

In [ ]:
import shutil as _shutil

for fname in ["metrics_report.txt", "qualitative_reconstruction.png", "sample_frames.png"]:
    src = os.path.join("/content/dense", fname)
    if os.path.exists(src):
        _shutil.copy2(src, os.path.join(DRIVE_SAVE_DIR, fname))

print("Saved metrics + qualitative figures to", DRIVE_SAVE_DIR)